# SETReliable — Full-Cohort Colab Runner

This notebook runs the repository's canonical SCMD pipeline from a pinned source commit. It starts from a clean generated-artifact state, checks the input and runtime, runs the full cohort without sampling, performs the saved-test empirical monotonicity audit, and packages reports.

**Important:** this notebook has not been executed as part of the code changes. Running the full cohort is a separate, potentially long Colab job. Inventory and stockout outputs remain simulated.

In [ ]:
import os, platform, shutil, subprocess, sys
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
print('CPU cores:', os.cpu_count())
print('Disk free (GB):', round(shutil.disk_usage('/content').free / 1024**3, 2))
if sys.version_info[:2] != (3, 11):
    raise RuntimeError(f'Python 3.11 is required; found {sys.version_info.major}.{sys.version_info.minor}')

In [ ]:
REPO_URL = 'https://github.com/Souptik-Hazra/SET_new.git'
REPO_COMMIT = 'dfb87e6015ab3c33d07302ce911616293b77e814'
REPO_DIR = Path('/content/SET_new')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print('PINNED_COMMIT:', REPO_COMMIT)
print('CHECKED_OUT_COMMIT:', actual_sha)
if actual_sha != REPO_COMMIT:
    raise RuntimeError('Repository SHA mismatch; refusing to run any pipeline stage.')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=REPO_DIR, check=True)
print('Dependencies installed from the pinned repository requirements.txt')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set this to the actual CSV path in your Google Drive.
SOURCE_CSV = Path('/content/drive/MyDrive/REPLACE_WITH_YOUR_SCMD_CSV.csv')
BOOTSTRAP_RESAMPLES = 2000

if not SOURCE_CSV.is_file():
    raise FileNotFoundError(f'Set SOURCE_CSV to your actual source CSV: {SOURCE_CSV}')
source_bytes = SOURCE_CSV.stat().st_size
free_bytes = shutil.disk_usage('/content').free
minimum_free_bytes = 10 * 1024**3 + 6 * source_bytes
print('SOURCE_CSV:', SOURCE_CSV)
print('CSV size (GB):', round(source_bytes / 1024**3, 3))
print('Disk free (GB):', round(free_bytes / 1024**3, 2))
if free_bytes < minimum_free_bytes:
    raise RuntimeError('Insufficient free disk under the conservative preflight estimate (6x CSV size + 10 GB).')

# DuckDB parses the source and reports actual CSV data rows before ingestion.
import duckdb
con = duckdb.connect()
source_rows = con.execute('SELECT COUNT(*) FROM read_csv_auto(?, header=true)', [str(SOURCE_CSV)]).fetchone()[0]
con.close()
print('SOURCE_DATA_ROWS:', source_rows)
if source_rows == 0:
    raise RuntimeError('Source CSV contains no data rows.')

## Clean run and full-cohort execution

The current `main` branch uses one canonical runner rather than the old notebook stage/checkpoint wrapper. This cell deletes only generated `data/processed/` and `output/` artifacts; it does not delete the source CSV. It does not sample or reduce the cohort.

In [ ]:
for relative in ('data/processed', 'output'):
    target = REPO_DIR / relative
    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable, 'scripts/run_pipeline.py',
    '--source', str(SOURCE_CSV),
    '--bootstrap', str(BOOTSTRAP_RESAMPLES),
]
print('EXECUTING:', ' '.join(command))
subprocess.run(command, cwd=REPO_DIR, check=True)
print('CANONICAL_PIPELINE_EXITED_SUCCESSFULLY')

In [ ]:
# Independently rerun the real saved-test audit after the integrated pipeline gate.
subprocess.run(
    [sys.executable, '-m', 'src.set_forecasting.validation.empirical_monotonicity'],
    cwd=REPO_DIR, check=True,
)

import json
audit_path = REPO_DIR / 'output' / 'empirical_monotonicity_audit.json'
audit = json.loads(audit_path.read_text(encoding='utf-8'))
print('EMPIRICAL_AUDIT_STATUS:', audit['status'])
print('TOTAL_TEST_OBSERVATIONS:', audit['total_test_observations'])
print('DISTRIBUTION_PASS/FAIL:', audit['distribution_pass'], audit['distribution_fail'])
print('RADCSR_PASS/FAIL:', audit['radcsr_pass'], audit['radcsr_fail'])
print('LOW_AVAILABILITY_FAILURES:', audit['low_availability_failures'])
print('PER_SCENARIO:', json.dumps(audit['per_scenario'], indent=2))
if audit['status'] != 'PASS' or audit['low_availability_failures']:
    raise RuntimeError('Empirical monotonicity audit failed; inspect output/empirical_monotonicity_audit.json')

In [ ]:
import pyarrow.parquet as pq
required_outputs = [
    'output/pipeline_report.json',
    'output/final_verdict.md',
    'output/empirical_monotonicity_audit.json',
    'output/radcsr_monotonicity_audit.json',
    'output/tables/forecast_metrics.csv',
    'output/tables/risk_brier_by_scenario.csv',
    'output/tables/risk_calibration_metrics.csv',
    'output/tables/radcsr_vs_baselines_bootstrap_holm.csv',
    'data/processed/scmd_monthly_series.parquet',
    'data/processed/forecasts.parquet',
    'data/processed/scenario_streams.parquet',
    'data/processed/simulated_inventory.parquet',
    'data/processed/radcsr_risk_details.parquet',
]
missing = [p for p in required_outputs if not (REPO_DIR / p).is_file() or (REPO_DIR / p).stat().st_size == 0]
if missing:
    raise RuntimeError(f'Missing or empty required artifacts: {missing}')
for relative in required_outputs:
    path = REPO_DIR / relative
    detail = f'{relative}: {path.stat().st_size:,} bytes'
    if path.suffix == '.parquet':
        detail += f' | rows={pq.ParquetFile(path).metadata.num_rows:,}'
    print(detail)

archive_base = Path('/content/SET_new_outputs')
archive = shutil.make_archive(str(archive_base), 'zip', root_dir=REPO_DIR, base_dir='output')
drive_target = Path('/content/drive/MyDrive/SET_new_outputs.zip')
shutil.copy2(archive, drive_target)
print('OUTPUT_ARCHIVE:', drive_target)